# Altitude Tuning on the Duckiedrone

In this notebook you will deploy the altitude PID controller built in previous notebooks onto a virtual and/or physical Duckiedrone, and tune the gains through live flight.

### How it works

The Duckiedrone (DD24-B) uses **PX4** as its flight controller and communicates with the companion computer over **MAVLink** via **MAVROS2** (ROS 2 Jazzy). Your altitude PID runs as a ROS 2 node and commands PX4 in **OFFBOARD mode** by publishing:

| Topic | Message | Purpose |
|---|---|---|
| `/mavros/setpoint_attitude/attitude` | `geometry_msgs/PoseStamped` | Level quaternion (keeps the drone horizontal) |
| `/mavros/setpoint_attitude/thrust` | `mavros_msgs/Thrust` | Normalized thrust in [0, 1], your PID output |

The bottom Time-of-Flight (ToF) rangefinder provides the altitude measurement.

### Conventions used below

Commands below use `DRONE` as a placeholder for your drone's name. Replace it with the actual name before running anything.

### Virtual (Sim) and physical (Real) Duckiedrone values are different

The virtual Duckiedrone and the physical DD24-B do not hover at the same thrust, so you should not expect the final numbers to be the same.

Your PID gains ($K$, $K_p$, $K_i$, $K_d$) in `z_pid.yaml`, similarly, end up different for the virtual and for the physical drone. The file keeps both sets side by side, with the sim values commented out, so check which set is active before you fly.

The thrust needed to hover is also different between the two. Keep this in mind throughout this LX: a starting point that works in simulation is not the right starting point on the physical drone, and vice versa.

### Thrust safety cap

Before your PID's thrust output reaches PX4, it passes through a hard ceiling set by the `thrust_cap` parameter (default `0.55`).

No matter what your gains compute, the Duckiedrone's thrust will never be commanded above this value. This is a safety feature for tuning. Higher values might cause sharp acceleration of the drone and potentially catastrophic outcomes.

Keep it in mind as you go: if the drone stops climbing or stops responding even after you raise a gain, you may be running into this cap rather than hitting a PID limitation. You will see a warning in the terminal logs whenever it kicks in. Modify the safety cap carefully, only if necessary, after understanding the implications.

## Step 0: Set Up Your Drone

Choose **one** of the two paths below, then continue to Step 1.

---

### Option A: Virtual drone (Duckiematrix)

Start here to validate your PID controller in simulation before flying on the physical Duckiedrone.

**A.1: Create and start the virtual drone**

Virtual robots are exact software replicas of the physical robots. In Duckietown, you need to `create` your virtual Duckiedrone only once, the first time you set it up. You can then manage your virtual robot fleet with `start` and `stop` commands. 

```bash
dts duckiebot virtual create -t duckiedrone -c DD24 DRONE #only first time
dts duckiebot virtual start DRONE 
```

Creating the virtual Duckiedrone might take several minutes, depending on the configuration of your machine. After creating DRONE, you can verify successful operation with:

```bash
dts duckiebot virtual list
```

Other virtual robot commands are summarized on the [Duckietown Manual](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/virtual-duckietown-robots/introduction-to-virtual-duckiebots.html). 

**A.2: Launch the Duckiematrix**

Assuming the Duckiematrix is already installed on your computer with `dts matrix install`, start it with:

```bash
dts matrix run --standalone --map sandbox_drone --embedded
```

> **macOS**: run `dts matrix engine run --embedded --map sandbox_drone` inside the Duckietown workspace, then `dts matrix run` in a separate terminal **outside** the workspace.


**A.3: Attach the drone to the map**

```bash
dts matrix attach DRONE map_0/vehicle_0
```
---

### Option B: Physical Duckiedrone

**B.1: Power on the Duckiedrone** and wait for the Raspberry Pi to boot.

**B.2: Verify connectivity**

```bash
ping DRONE.local
```

**B.3: SSH into the drone**

```bash
ssh duckie@DRONE.local
```

Password: `quackquack`

**B.4: Confirm the drone stack is running**

The ROS 2 environment lives inside the drone's Docker containers, not directly in the SSH shell you just opened.

```bash
docker exec -it ros2-mavros bash
```

From inside that container, run:

```bash
ros2 topic echo /mavros/state
```

Expect the output to show this (emphasis on **connected: true**):
```text
header:
  stamp:
    sec: 1786270817
    nanosec: 849732881
  frame_id: ''
connected: true
armed: false
guided: true
manual_input: false
mode: AUTO.LOITER
system_status: 0
```

Press `Ctrl-C` to stop echoing the `mavros/state` messages and return to the terminal.

**B.5: Safety checklist**

- [ ] The drone is on a flat, unobstructed surface with at least 1 m of clear space above.
- [ ] Your `z_pid.yaml` has two separate gain sections, one for sim and one for the real drone. Make sure you have the **physical/real drone** section active, not the virtual/simulated one, they hover at different thrust so using the wrong one is unsafe.

## Step 1: Build, Deploy, and Launch

### 1.0 — Make sure your Duckiedrone is up to date

From a terminal on your base station, run:

```bash
dts duckiebot update DRONE
```

This process might take up to 10 minutes. 

### 1.1 — Build and deploy the LX code

From the LX root (`lx-dd-altitude-pid-control/`) on your laptop:

```bash
dts code build -R DRONE
```

> **Warning**: the command below (Step 1.2) starts the PID controller node. Once it runs, it will arm the drone and spin the motors, first at idle thrust, then at full PID thrust once you enable altitude control in Step 2.
>
> Before running it, make sure your drone is in a safe-to-fly condition: on a flat surface, with clear space above it, and with nothing near the propellers.

### 1.2 — Launch the workbench

```bash
dts code workbench -R DRONE
```

This runs the default launcher, which starts the PID controller node. Keep this terminal open.

### 1.3 — Open an interactive ROS 2 shell

In a **second terminal**, attach a shell to the running workbench container:

```bash
dts code workbench -R DRONE --shell
```

This gives you a fully sourced ROS 2 environment where you can run `ros2 topic`, `ros2 param`, `ros2 service`, etc.  
**All `ros2` commands in the rest of this notebook should be run in this shell.**

### 1.4 — Verify the rangefinder

In the shell from 1.3, confirm the bottom ToF sensor is publishing:

```bash
ros2 topic hz /DRONE/bottom_tof_driver_node/range    # should show ~30 Hz
```

### 1.5 — Watch the PID controller start up

Back in the **first terminal** (1.2), you will see the node go through two stages:

1. **PREFLIGHT** — streams 100 idle setpoints (thrust = 0.0) at 20 Hz.
2. **OFFBOARD_IDLE** — requests OFFBOARD mode and arms the vehicle. Thrust = 0.1 (below hover, drone stays on the ground).

Log messages indicate each stage as it completes.
You should see:
```
[pid_controller_node-1] [INFO] [1781786079.775730970] [altitude_pid_node]: Preflight complete, MAVROS connected — transitioning to OFFBOARD_IDLE.
[pid_controller_node-1] [INFO] [1781786079.826796908] [altitude_pid_node]: OFFBOARD mode requested successfully.
[pid_controller_node-1] [INFO] [1781786084.852122028] [altitude_pid_node]: Vehicle armed successfully.
[pid_controller_node-1] [INFO] [1781786085.526747951] [altitude_pid_node]: Vehicle just armed — PID state reset.
```

## Step 2: Enable Altitude Control

The PID **does not activate automatically**. In your **ROS 2 shell** (Step 1.3), explicitly enable it after confirming the node has reached OFFBOARD_IDLE:

```bash
ros2 service call /enable_altitude_control std_srvs/srv/Trigger
```

You should see:

```
success=True message='Altitude PID control enabled.'
```

The PID is now computing thrust from the rangefinder and commanding PX4. The drone will take off and attempt to hold the setpoint altitude.

> **Safety**: calling the same service again **disables** the PID and reverts to idle thrust.
>
> On a **physical drone**, always be ready to press **Ctrl-C** in the terminal where you launched the workbench (Step 1.2). This initiates a force-disarm. The node sends a kill command to PX4 and the motors stop once PX4 acknowledges it, so timing depends on the MAVLink link, not instant, but fast. It is your emergency stop; use it any time something looks wrong.
>
> The `thrust_cap` parameter (Step 3.2) also limits how much thrust the PID can ever demand, but it is a backup, not a replacement for keeping your hand ready on Ctrl-C.

## Step 3: Monitor and Tune Gains

All commands below are run in your **ROS 2 shell** (Step 1.3).

### 3.1 — Read the controller's log

While altitude control is active, the node prints its own internal state twice a second in the terminal from Step 1.2:

```
[pid_controller_node-1] [INFO] [1785744445.411206944] [altitude_pid_node]: alt=+0.252 sp=+0.250 err=-0.002 | P=-0.000 I=+0.001 D=+0.000 K=+0.700 | raw=0.700 cmd=0.700
```
> **Note**: the log appears only while altitude control is enabled. In `OFFBOARD_IDLE` the PID is not running, so there is nothing to report.

This is the control equation from [Notebook 3](./3-altitude_pid_activity.ipynb) evaluated once per cycle, with every term shown separately:

| Field | Meaning |
|---|---|
| `alt` | Measured altitude from the rangefinder (m) |
| `sp` | Target altitude, the `setpoint_z` parameter (m) |
| `err` | `sp - alt`. **Positive means the drone is below the target** and needs more thrust |
| `P` | The proportional term, $K_p \cdot e$ |
| `I` | The integral term, $K_i \cdot \sum e \, \Delta t$ |
| `D` | The derivative term, $K_d \cdot \dot{e}$ |
| `K` | The hover thrust offset. Constant, and it does not respond to error |
| `raw` | The PID output, `P + I + D + K`, before the thrust cap |
| `cmd` | The normalized thrust actually sent to PX4, `min(thrust_cap, raw)` |

The altitude and the commanded thrust can also be watched on their source topics:

```bash
ros2 topic echo /DRONE/bottom_tof_driver_node/range   # altitude (m)
ros2 topic echo /mavros/setpoint_attitude/thrust      # thrust command [0, 1]
```

For a live plot:

```bash
dts duckiebot graph_plotter DRONE
```

Subscribe to `node/tof_driver_bottom/out/range` in the plotter to watch altitude.

### 3.2 — Tune gains at runtime

All PID gains and the altitude setpoint are exposed as ROS 2 parameters. You can change them **without restarting the node**:

```bash
ros2 param set /altitude_pid_node kp 0.10
ros2 param set /altitude_pid_node ki 0.05
ros2 param set /altitude_pid_node kd 0.1
ros2 param set /altitude_pid_node k 0.4
ros2 param set /altitude_pid_node setpoint_z 0.5
ros2 param set /altitude_pid_node thrust_cap 0.60
```

To check a current value:

```bash
ros2 param get /altitude_pid_node kp
```

> **WARNING — Disarm before changing gains!**
>
> Changing gains mid-flight can cause sudden, dangerous thrust changes (especially large jumps in `k` or `kp`). **Always disarm the drone first:**
>
> ```bash
> ros2 service call /mavros/cmd/arming mavros_msgs/srv/CommandBool "{value: false}"
> ```
>
> After updating the gains, re-arm and re-enable altitude control:
>
> ```bash
> ros2 service call /mavros/cmd/arming mavros_msgs/srv/CommandBool "{value: true}"
> ros2 service call /enable_altitude_control std_srvs/srv/Trigger
> ```

### 3.3 — Recommended tuning workflow

1. **Fly** — enable altitude control, observe the response.
2. **Disable** — `ros2 service call /enable_altitude_control std_srvs/srv/Trigger`
3. **Disarm** — `ros2 service call /mavros/cmd/arming mavros_msgs/srv/CommandBool "{value: false}"`
4. **Adjust** — `ros2 param set /altitude_pid_node kp 0.25` (change one gain at a time)
5. **Re-arm & re-enable** — fly with the new gains immediately.
6. **Persist** — once satisfied, save the final values to `z_pid.yaml` so they survive a restart.

## Step 4: Ziegler-Nichols Tuning

You saw an overview of the Ziegler-Nichols method in [Notebook 1](./1-PID_control_overview.ipynb). Here is how to apply it on the DD24.

### Find the ultimate gain $K_u$

1. Start with $K_i = 0$, $K_d = 0$, $K_p = 0.1$, and $K$ set to your hover offset from `z_pid.yaml`.
2. Fly, observe, then disarm and increase $K_p$ using `ros2 param set`.
3. Repeat until the drone oscillates with **sustained, uniform amplitude** (not growing, not decaying).
4. Record this value as $K_u$ (the *ultimate gain*).

> **Note**: use the `K` value for the drone you are flying on right now.
>
> Sim and real do not hover at the same thrust, so their `K` values are different (see the note at the top of this notebook). Starting from the wrong one means you are not really starting from hover.

> **Tip**: if oscillations grow, $K_p$ is too high; if they decay, it is too low.

> **Watch out**: if $K$ is already close to `thrust_cap`, raising $K_p$ may just clip against the cap instead of producing a real oscillation.
>
> If the drone seems to stop responding as you raise $K_p$, check how close you are to the cap before assuming you found $K_u$.

### Measure the ultimate period $T_u$

5. While oscillating uniformly, use `dts duckiebot graph_plotter DRONE` (or `ros2 topic echo`) to capture altitude over time.
6. Measure the time between two consecutive peaks. Record this as $T_u$ (seconds).

### Compute PID gains

Use the Ziegler-Nichols formulas (classic PID):

$$K_p = 0.6\,K_u \qquad K_i = \frac{2\,K_p}{T_u} \qquad K_d = \frac{K_p\,T_u}{8}$$

Apply them with `ros2 param set` and test.

**Exercises**

1. Record your $K_u$ and $T_u$ values.
2. Compute and record the Ziegler-Nichols gains.
3. Fly with the Z-N gains. How well does the drone hold the 0.25 m setpoint? (Z-N is a starting point, not a final answer.)

## Step 5: Fine-Tuning

Empirically refine the gains from Step 4.

**Guidelines** (adjust one gain at a time):

| Symptom | Action |
|---|---|
| Overshoot / slow convergence | Adjust $K_p$/$K_d$ ratio |
| Steady-state offset (hovers above/below setpoint) | Increase $K_i$ |
| Oscillation | Reduce $K_p$ or increase $K_d$ |
| Slow response | Increase $K_p$ |

### Change the altitude setpoint

You can also change the target altitude on the fly:

```bash
ros2 param set /altitude_pid_node setpoint_z 0.5
```

### Attitude override (optional)

While the altitude PID is active, you can nudge the drone horizontally by publishing an attitude override:

```bash
ros2 topic pub --once /dd24/attitude_override geometry_msgs/msg/PoseStamped \
  '{pose: {orientation: {x: -0.05, y: 0.0, z: 0.0, w: 0.9987}}}'
```

The override expires after 0.5 s — the node reverts to level hover automatically.

**Exercises**

1. Tune your PID gains so the drone holds the 0.25 m setpoint with minimal oscillation and fast convergence.
2. Save your final values to `z_pid.yaml`.
3. Change the setpoint to 0.5 m. Does the controller generalize?

<!--
## Handin

Record your answers to all exercises in `answers_pid.md` inside `packages/solution/solution/` and push your final code to GitHub.

Include:
- Final values of $K, K_p, K_i, K_d$
- Your $K_u$ and $T_u$ measurements
- Brief description of how each gain change affected the drone's behavior
-->